In [1]:
import os
while not os.path.exists('METHODOLOGY.md') and os.getcwd() != '/': os.chdir('..')
assert os.path.exists('METHODOLOGY.md'), 'run from inside the Nofit_LRT_Extension repository'

# Step 45 — Demand for the LRT alternatives: main route + extension and extension only, two regimes, 2022 / 2040 / 2050 × BU / HS, AM and PM, at TAZ and corridor-area level (`LRT_alternatives_demand.ipynb`)

The **main Nofit LRT route** (Hamifrats – Nazareth, under construction; `Input/Main_Nofit/`, 49
Emme links, 40.7 km) joined the repository on 5 October 2026 beside the **extension** this study
estimates demand for (`Input/GeneralHalufa/hf_lrt_3`, Tirat Carmel – Hamifrats, 18.9 km, 24
stations, step 25). This step estimates the LRT demand for two **alternatives** — (1) the main
route and the extension as one through line, (2) the extension alone — in two **regimes** for the
extension — **Prioritized**, an LRT with full priority at the interchanges, represented by step
25's underground-calibrated running times, and **Unprioritized**, at grade with step 25's ground
times (no underground line is planned; the two regimes differ in running time only, the station
access being the same) — with the main route at its **80 km/h design speed**; for **2022** and the four
forecast sets **BU / HS × 2040 / 2050**; for the **AM** (06:00–09:00) and **PM** (16:00–19:00)
periods; at **TAZ** level (the 174 corridor TAZs) and aggregated to the **25 corridor areas**;
both directions of the line. The out-of-vehicle parameters are the **central set of the OVT
research** (§6al, `Output/ovt_research/parameter_summary.csv`), and the LRT carries a premium over
the bus and a smaller, estimated premium over the Metronit BRT.

**Method in one paragraph.** The level of service is built at TAZ level as in step 44 (§6ao):
the car on the May 2026 speeds between TAZ centroids; the transit as a frequency-based line graph
of the GTFS bus and Metronit services of the period with step 30's observed running times. The
LRT is added to that graph as station nodes reached on foot (from the TAZ or from nearby stops) or
by a bus / Metronit feeder leg found by the same shortest path: the LRT cost of a TAZ pair is the
cheapest station pair's access leg + boarding penalty + wait + station-access term + LRT running
time + egress leg. Demand follows step 31's pivot per TAZ pair (§6ac, §6ap): the LRT's share of the
transit nest from the LRT – bus cost gap, the car → transit shift from the nest's cost gain,
pivoted on each TAZ pair's observed share smoothed towards its area pair's. The 2022 base is the
step-22 AM layers and the wave-1b PM layers; the forecast AM sets are step 23's, and the PM sets
scale the PM base by step 23's AM growth per TAZ pair (a stated approximation: step 23's margins
are AM-specific).

**Stated assumptions.** (a) The main route's **stations are the delivered stop nodes**
(`Input/Main_Nofit/Main_Nofit_StopsID.csv`, corrected list of 5 October 2026): 20 of the route's
50 nodes, 2.1 km apart on average, the first of them the junction node at Hamifrats (the first
version of this step, before the list, had assumed 27 stations at ≥ 1 km). (b) Main-route times:
80 km/h between stops + 10 s dwell per stop (the design-regime form of step 25), at grade. (c) One line through-runs at a
5-minute headway (wait 2.5 min), no transfer at Hamifrats. (d) **LRT premium over the Metronit:
2.5 generalized minutes** (range 1.5–3.5), derived from the calibrated models' in-vehicle
multipliers in the OVT evidence table — LRT 0.80–0.85, BRT 0.90–0.95 — scaled to the model's
5-minute LRT-over-bus premium (a 0.15 multiplier gap ≈ 5 min, a 0.05–0.10 gap ≈ 1.7–3.3 min).
(e) OVT central: walk 2.0, wait 2.0, bus ↔ LRT 8, BRT ↔ LRT 4; station access **0.5 min per end in
both regimes** — "Prioritized" stands for an LRT with priority at the interchanges, not an
underground line, so the research's 1.5-minute underground term does not apply (decision of
5 October 2026); walking at **4 km/h** (66.7 m/min) on the straight line × 1.3 (decision of the
same day; the chain's other steps use 4.8 km/h). (f) λ = 0.03, λ_T = 0.06 (step 31's central).

In [2]:
import time, zipfile, json
import numpy as np
import pandas as pd
import geopandas as gpd
import shapefile
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from shapely.geometry import LineString, Point
from shapely.strtree import STRtree
from scipy.sparse import coo_matrix, csr_matrix
from scipy.sparse.csgraph import dijkstra, connected_components
from scipy.spatial import cKDTree

BLUE, ORANGE, AQUA, PURPLE, INK, INK2, MUTED, GRID, AXIS = '#2a78d6', '#eb6834', '#1baf7a', '#7b5bd6', '#0b0b0b', '#52514e', '#898781', '#e1e0d9', '#c3c2b7'
OUT = 'Output/alternatives'; FIG = 'Output/figures'; os.makedirs(OUT, exist_ok=True); os.makedirs(FIG, exist_ok=True)
# --- parameters ---
W_WALK, W_WAIT, TRANSFER_PEN, BRT_LRT_PEN = 2.0, 2.0, 8.0, 4.0                               # OVT central (Output/ovt_research/parameter_summary.csv, proposed_central)
ACC_STATION = 0.5                                                                             # station access per end, both regimes (Prioritized = priority at interchanges, not underground; 5 Oct 2026)
WALK_SPEED, DETOUR, WAIT_CAP = 4.0 * 1000 / 60, 1.3, 10.0                                     # 4 km/h = 66.7 m/min (decision of 5 Oct 2026); straight line × 1.3
LAM, LAM_T, PREM_BUS, PREM_BRT, K_EB = 0.03, 0.06, 5.0, 2.5, 20.0                              # step 31's central λ; LRT premium over the bus 5; over the Metronit 2.5 (estimated, see above)
LRT_HEADWAY = 5.0; LRT_WAIT = LRT_HEADWAY / 2
MAIN_SPEED_KMH, MAIN_DWELL_S, MAIN_MIN_SPACING_M = 80.0, 10.0, 1000.0
ACCESS_M, TRANSFER_M, STATION_STOP_M = 1000.0, 300.0, 400.0
CAR_FALLBACK_KMH, CAR_CONNECTOR_KMH, CAR_TERMINAL_MIN, MATCH_TOL = 30.0, 20.0, 3.0, 3.0
PERIODS = {'AM': {'hours': (6, 9), 'peak': (7, 8), 'car_hour': 7, 'label': 'AM 06:00–09:00'}, 'PM': {'hours': (16, 19), 'peak': (17, 18), 'car_hour': 17, 'label': 'PM 16:00–19:00'}}
SCEN_YEARS = ['2022', 'BU_2040', 'BU_2050', 'HS_2040', 'HS_2050']
ALTS = {'main_ext': 'Main route + extension', 'ext': 'Extension only'}
REGIMES = {'prioritized': 'Prioritized (priority at interchanges; step 25 underground-calibrated times)', 'unprioritized': 'Unprioritized (at grade; step 25 ground times)'}
# --- zones ---
xl = pd.ExcelFile('Input/Corridor_TAZ_Agg_V2.xlsx'); areas = xl.parse('AreaCodes').set_index('AggCode'); key = xl.parse('TazAgg')
AREAS = list(areas.index); names = areas['AggAreaName']; A = len(AREAS); apos = {a: i for i, a in enumerate(AREAS)}
tzg = gpd.read_file('Input/TAZ_North/TAZ_North.shp', engine='pyogrio')
TAZC = pd.DataFrame({'TAZ': tzg['TAZ_NUMBER'].astype(int), 'cx': tzg.geometry.centroid.x, 'cy': tzg.geometry.centroid.y}).drop_duplicates('TAZ').set_index('TAZ')
TAZS = sorted(key['TAZ'].astype(int)); NT = len(TAZS); tpos = {t: i for i, t in enumerate(TAZS)}
ta = key.set_index('TAZ')['AggCode'].reindex(TAZS).astype(int).values; tai = np.array([apos[a] for a in ta]); TC = TAZC.loc[TAZS, ['cx', 'cy']].values.astype(float)
zon = pd.read_csv('Input/Zonal_2020.csv', encoding='windows-1255').set_index('TAZ_ID'); pop = zon['POPULATION'].reindex(TAZS).fillna(0).values; emp = zon['EMPL_TOT'].reindex(TAZS).fillna(0).values
INTER = ta[:, None] != ta[None, :]
print(f'{NT} corridor TAZs in {A} areas; OVT central walk {W_WALK} / wait {W_WAIT} / transfer {TRANSFER_PEN} / BRT–LRT {BRT_LRT_PEN} / station access {ACC_STATION} per end (both regimes); walking {WALK_SPEED * 60 / 1000:.1f} km/h; λ {LAM} / λ_T {LAM_T}; premium over bus {PREM_BUS}, over BRT {PREM_BRT}')

174 corridor TAZs in 25 areas; OVT central walk 2.0 / wait 2.0 / transfer 8.0 / BRT–LRT 4.0 / station access 0.5 per end (both regimes); walking 4.0 km/h; λ 0.03 / λ_T 0.06; premium over bus 5.0, over BRT 2.5


## 1. The two lines: stations and running times

The extension's 24 stations and the two calibrated time matrices are step 25's. The main route
is chained from its 49 links into one path from node 34000 — 103 m from the extension's S24 at
Hamifrats, so the two lines meet there — to node 34034 in Nazareth; its stations are the 20
delivered stop nodes, in path order; its station-to-station times are the chainage at 80 km/h
plus 10 s per intermediate stop. The through line of alternative 1 is S01 … S24 = M01 … M20, its
time matrix the sum of the two legs through S24.

In [3]:
st_ext = pd.read_csv('Output/lrt_v2/lrt_stations_hf_lrt_3.csv'); st_ext = st_ext[['station_id', 'chainage_m', 'x', 'y', 'TAZ', 'AggCode']].copy()
T_EXT = {r: pd.read_csv(f'Output/lrt_v2/lrt_station_times_{f}.csv', index_col=0) for r, f in (('prioritized', 'all_underground'), ('unprioritized', 'all_ground'))}
m = gpd.read_file('Input/Main_Nofit/Main_Nofit 2026-10-05.shp', engine='pyogrio').to_crs(2039)
adj = {}
for r in m.itertuples(): adj.setdefault(r.AB_INODE, []).append((r.AB_JNODE, r.Index)); adj.setdefault(r.AB_JNODE, []).append((r.AB_INODE, r.Index))
path, used, cur = [34000], set(), 34000
while True:
    nxt = [(n, i) for n, i in adj[cur] if i not in used]
    if not nxt: break
    n, i = nxt[0]; used.add(i); path.append(n); cur = n
coords = {}
for r in m.itertuples(): coords[r.AB_INODE] = r.geometry.coords[0]; coords[r.AB_JNODE] = r.geometry.coords[-1]
ch = [0.0]
for a, b in zip(path[:-1], path[1:]): ch.append(ch[-1] + m.geometry.iloc[[i for n, i in adj[a] if n == b][0]].length)
main_nodes = pd.DataFrame({'node': path, 'x': [coords[n][0] for n in path], 'y': [coords[n][1] for n in path], 'chainage_m': ch})
# the stations: the stop nodes delivered on 5 October 2026 (Input/Main_Nofit/Main_Nofit_StopsID.csv, node ids of AB_INODE / AB_JNODE), in path order
stops_id = pd.read_csv('Input/Main_Nofit/Main_Nofit_StopsID.csv'); stop_set = set(stops_id['Node'].astype(int))
missing = stop_set - set(path); assert not missing, f'stop nodes not on the chained route: {missing}'
keep = [i for i, n in enumerate(path) if n in stop_set]
if 0 not in keep: keep = [0] + keep                                            # node 34000 is the junction with S24; it is a stop in the list
st_main = main_nodes.loc[keep].reset_index(drop=True); st_main['station_id'] = [f'M{k + 1:02d}' for k in range(len(st_main))]
print(f'main-route stops from the delivered list: {len(stop_set)} nodes, all on the chained route; {len(st_main)} stations M01 … M{len(st_main):02d}')
tz_pts = gpd.GeoSeries([Point(x, y) for x, y in zip(st_main['x'], st_main['y'])], crs=2039); jn = gpd.sjoin(gpd.GeoDataFrame(geometry=tz_pts), tzg[['TAZ_NUMBER', 'geometry']], how='left', predicate='within')
st_main['TAZ'] = jn.groupby(jn.index)['TAZ_NUMBER'].first().reindex(st_main.index).values; st_main['AggCode'] = key.set_index('TAZ')['AggCode'].reindex(st_main['TAZ']).values
seg_km = np.diff(st_main['chainage_m'].values) / 1000; sec_min = seg_km / MAIN_SPEED_KMH * 60 + MAIN_DWELL_S / 60
cum = np.concatenate([[0], np.cumsum(sec_min)]); T_MAIN = pd.DataFrame(np.abs(cum[:, None] - cum[None, :]), index=st_main['station_id'], columns=st_main['station_id'])
d_join = np.hypot(st_main.loc[0, 'x'] - st_ext.iloc[-1]['x'], st_main.loc[0, 'y'] - st_ext.iloc[-1]['y'])
print(f"main route: {len(main_nodes)} nodes, {ch[-1] / 1000:.1f} km, {len(st_main)} stations from the delivered stop list (mean spacing {ch[-1] / (len(st_main) - 1) / 1000:.2f} km); M01 is {d_join:.0f} m from S24; "
      f"end to end at {MAIN_SPEED_KMH:.0f} km/h + {MAIN_DWELL_S:.0f} s per stop: {cum[-1]:.1f} min ({ch[-1] / 1000 / cum[-1] * 60:.1f} km/h commercial)")
print('extension end to end: ' + ', '.join(f"{r} {T_EXT[r].iloc[0, -1]:.1f} min" for r in T_EXT))
# station tables per alternative; through-line time matrices per regime (M01 ≡ S24: the main leg's times are measured from M01)
STATIONS = {}; T_LINE = {}
st_main_rest = st_main.iloc[1:].copy()
for alt in ALTS:
    s = pd.concat([st_ext.assign(line='extension', surface=False), st_main_rest.assign(line='main', surface=True)[['station_id', 'chainage_m', 'x', 'y', 'TAZ', 'AggCode', 'line', 'surface']]]) if alt == 'main_ext' else st_ext.assign(line='extension', surface=False)
    s = s.reset_index(drop=True); s['order'] = range(len(s)); s['chainage_line_m'] = np.where(s['line'] == 'main', s['chainage_m'] + st_ext.iloc[-1]['chainage_m'], s['chainage_m']); STATIONS[alt] = s
    for reg in REGIMES:
        ids = list(s['station_id']); T = pd.DataFrame(0.0, index=ids, columns=ids)
        te = T_EXT[reg].reindex(index=st_ext['station_id'], columns=st_ext['station_id']); T.loc[te.index, te.columns] = te.values
        if alt == 'main_ext':
            tm = T_MAIN.reindex(index=st_main['station_id'], columns=st_main['station_id']); mids = list(st_main_rest['station_id'])
            T.loc[mids, mids] = tm.loc[mids, mids].values
            via = te.loc[:, 'S24'].values[:, None] + tm.loc['M01', mids].values[None, :]      # extension station -> S24 -> main station
            T.loc[te.index, mids] = via; T.loc[mids, te.index] = via.T
        T_LINE[(alt, reg)] = T
        s[f'surface_{reg}'] = np.where(s['line'] == 'main', True, reg == 'unprioritized')
    s.to_csv(f'{OUT}/stations_{alt}.csv', index=False, float_format='%.1f')
for (alt, reg), T in T_LINE.items(): T.to_csv(f'{OUT}/lrt_station_times_{alt}_{reg}.csv', float_format='%.2f')
print('through line (alternative 1), S01 → Nazareth: ' + ', '.join(f"{reg} {T_LINE[('main_ext', reg)].iloc[0, -1]:.1f} min" for reg in REGIMES))
main_nodes.to_csv(f'{OUT}/main_route_nodes.csv', index=False, float_format='%.1f')

main-route stops from the delivered list: 20 nodes, all on the chained route; 20 stations M01 … M20


main route: 50 nodes, 40.7 km, 20 stations from the delivered stop list (mean spacing 2.14 km); M01 is 103 m from S24; end to end at 80 km/h + 10 s per stop: 33.7 min (72.5 km/h commercial)
extension end to end: prioritized 40.6 min, unprioritized 65.8 min
through line (alternative 1), S01 → Nazareth: prioritized 74.3 min, unprioritized 99.6 min


/tmp/ipykernel_2683/607598765.py:23: UserWarning: CRS mismatch between the CRS of left geometries and the CRS of right geometries.
Use `to_crs()` to reproject one of the input geometries to match the CRS of the other.

Left CRS: EPSG:2039
Right CRS: PROJCS["GRS_1980_Transverse_Mercator",GEOGCS["Unkn ...

  tz_pts = gpd.GeoSeries([Point(x, y) for x, y in zip(st_main['x'], st_main['y'])], crs=2039); jn = gpd.sjoin(gpd.GeoDataFrame(geometry=tz_pts), tzg[['TAZ_NUMBER', 'geometry']], how='left', predicate='within')


## 2. Time on route: LRT, Metronit, train and car

End-to-end and segment times on the corridor, per period where the mode has one: the LRT from
the matrices above; the **Metronit** line 1 (route code 83001, Hof HaCarmel – Krayot) from its
scheduled trips of the period and step 30's observed running times; the **train** between the
corridor's Israel Railways stations (Hof HaCarmel – Kiryat Motzkin) from the day's rail trips;
the **car** on step 37's network at the period's hour between the corridor areas' points
(Tirat Carmel → Hamifrats → Nazareth), plus the 3-minute terminal.

In [4]:
Z = zipfile.ZipFile('Input/GTFS/israel-public-transportation.zip')
gstops = pd.read_csv(Z.open('stops.txt'), usecols=['stop_id', 'stop_name', 'stop_lat', 'stop_lon'])
stops_sa = pd.read_csv('Output/gtfs/stops_study_area.csv').set_index('stop_id')
ST = {p: pd.read_csv(f'Output/gtfs/stop_times_study_area_{p.lower()}_trips_v45.csv.gz') for p in PERIODS}
rail = pd.read_csv('Output/gtfs/stop_times_study_area_rail_day.csv.gz')
seg_obs = pd.read_csv('Output/gtfs/bus_segments_observed.csv'); seg_obs = seg_obs[seg_obs['flag'].isin(['ok', 'ok_undirected']) & (seg_obs['observed_min'] > 0)].set_index(['stop_a', 'stop_b'])['observed_min']
route_rows = []
def trip_time_stats(df, label, period):
    """end-to-end scheduled time of the trips in df (first to last study-area stop), by direction: median, and the observed-time ratio of step 30 where its segments cover the trips"""
    g = df.sort_values(['trip_id', 'stop_sequence']).groupby('trip_id')
    ends = g.agg(first_stop=('stop_id', 'first'), last_stop=('stop_id', 'last'), t0=('dep', 'first'), t1=('arr', 'last'), n=('stop_id', 'size'), d=('direction_id', 'first'))
    ends['sched_min'] = (ends['t1'] - ends['t0']) / 60
    for d, e in ends.groupby('d'):
        e = e[e['n'] >= e['n'].max() * 0.8]
        if len(e) == 0: continue
        fs = gstops.set_index('stop_id')['stop_name']
        route_rows.append({'mode': label, 'period': period, 'direction': int(d), 'from': fs.get(e['first_stop'].mode().iloc[0], ''), 'to': fs.get(e['last_stop'].mode().iloc[0], ''), 'trips': len(e), 'stops': int(e['n'].median()),
                           'scheduled_min (median)': e['sched_min'].median(), 'observed_min (step 30 ratio)': np.nan})
for p, cfg in PERIODS.items():
    brt = ST[p][ST[p]['is_brt'] & (ST[p]['route_code'] == 83001)]; trip_time_stats(brt, 'Metronit line 1 (83001)', p)
    rp = rail[rail['dep'].between(cfg['hours'][0] * 3600, cfg['hours'][1] * 3600)]; rp = rp[rp['trip_id'].isin(rp['trip_id'])]
    trip_time_stats(rail[rail['trip_id'].isin(set(rp['trip_id']))], 'Train (Israel Railways, study-area stations)', p)
# observed ratio for the Metronit from step 30's per-trip table
obs = pd.read_csv('Output/gtfs/bus_trips_observed.csv.gz'); ob = obs[obs['route_code'] == 83001]
ratio_brt = (ob['observed_min'].sum() / ob['scheduled_min'].sum()) if len(ob) else np.nan
for r_ in route_rows:
    if r_['mode'].startswith('Metronit'): r_['observed_min (step 30 ratio)'] = r_['scheduled_min (median)'] * ratio_brt
# the train on the corridor itself: Hof HaCarmel <-> Kiryat Motzkin, per rail trip of the period that calls at both
rail_names = gstops.set_index('stop_id')['stop_name']; rail['name'] = rail['stop_id'].map(rail_names)
for p, cfg in PERIODS.items():
    rp = rail[rail['trip_id'].isin(set(rail[rail['dep'].between(cfg['hours'][0] * 3600, cfg['hours'][1] * 3600)]['trip_id']))]
    for a_n, b_n in (('חוף הכרמל', 'קרית מוצקין'), ('קרית מוצקין', 'חוף הכרמל')):
        tt = []
        for tid, g_ in rp.groupby('trip_id'):
            ga, gb = g_[g_['name'] == a_n], g_[g_['name'] == b_n]
            if len(ga) and len(gb) and gb['stop_sequence'].iloc[0] > ga['stop_sequence'].iloc[0]: tt.append((gb['arr'].iloc[0] - ga['dep'].iloc[0]) / 60)
        if tt: route_rows.append({'mode': 'Train (Israel Railways, corridor segment)', 'period': p, 'direction': f'{a_n} → {b_n}', 'from': a_n, 'to': b_n, 'trips': len(tt), 'stops': np.nan, 'scheduled_min (median)': float(np.median(tt)), 'observed_min (step 30 ratio)': np.nan})
# LRT rows
for (alt, reg), T in T_LINE.items():
    s = STATIONS[alt]; last = s['station_id'].iloc[-1]
    for a, b, lab in [('S01', 'S24', 'Tirat Carmel (S01) → Hamifrats (S24)')] + ([('S24', last, f'Hamifrats (S24) → Nazareth ({last})'), ('S01', last, f'Tirat Carmel (S01) → Nazareth ({last})')] if alt == 'main_ext' else []):
        route_rows.append({'mode': f'LRT — {ALTS[alt]}, {REGIMES[reg]}', 'period': 'all', 'direction': 'both (symmetric)', 'from': lab.split(' → ')[0], 'to': lab.split(' → ')[1], 'trips': np.nan, 'stops': int(s['order'].max() + 1),
                           'scheduled_min (median)': T.loc[a, b], 'observed_min (step 30 ratio)': np.nan})
# car: step 37's network, hour 7 / 17, between area points along the corridor
car_net = gpd.read_file('Output/car_speed/GoogleSpeed_202605_North.gpkg', engine='pyogrio')
BB = tzg.total_bounds; rd = shapefile.Reader('Input/BusSpeedData/Streets/Streets'); segs = []
for s_, rec in zip(rd.iterShapes(), rd.iterRecords()):
    b = s_.bbox
    if not (b[2] > BB[0] and b[0] < BB[2] and b[3] > BB[1] and b[1] < BB[3]): continue
    segs.append((LineString(s_.points), rec['DIR']))
tree = STRtree(car_net.geometry.values); G_car, L_car = car_net.geometry.values, car_net.geometry.length.values
match = np.full(len(segs), -1); same = np.ones(len(segs), bool)
for k, (L, dr) in enumerate(segs):
    a, b = Point(L.coords[0]), Point(L.coords[-1])
    for c in tree.query(L.buffer(MATCH_TOL)):
        g = G_car[c]
        if g.distance(a) < MATCH_TOL and g.distance(b) < MATCH_TOL and L.length <= L_car[c] + MATCH_TOL: match[k] = c; same[k] = g.project(a) < g.project(b); break
mm = match >= 0; idx = np.where(mm, match, 0)
E = pd.DataFrame({'ax': [s_[0].coords[0][0] for s_ in segs], 'ay': [s_[0].coords[0][1] for s_ in segs], 'bx': [s_[0].coords[-1][0] for s_ in segs], 'by': [s_[0].coords[-1][1] for s_ in segs], 'len': [s_[0].length for s_ in segs], 'dir': [s_[1] for s_ in segs]})
for h in sorted({cfg['car_hour'] for cfg in PERIODS.values()}):
    ab, ba = car_net[f'SPD_{h}_AB'].fillna(0).values[idx], car_net[f'SPD_{h}_BA'].fillna(0).values[idx]
    E[f'v{h}_AB'] = np.where(mm, np.where(same, ab, ba), 0.0); E[f'v{h}_BA'] = np.where(mm, np.where(same, ba, ab), 0.0)
cnodes = {}
def cnode(x, y):
    k = (round(x), round(y))
    if k not in cnodes: cnodes[k] = len(cnodes)
    return cnodes[k]
E['a'] = [cnode(x, y) for x, y in zip(E['ax'], E['ay'])]; E['b'] = [cnode(x, y) for x, y in zip(E['bx'], E['by'])]; NC = len(cnodes); CXY = np.array([k for k, v in sorted(cnodes.items(), key=lambda kv: kv[1])], dtype=float)
def car_graph(h):
    rows, cols, tmin, lens = [], [], [], []
    for a, b, ln, dr, vab, vba in zip(E['a'], E['b'], E['len'], E['dir'], E[f'v{h}_AB'], E[f'v{h}_BA']):
        for (u, w, v, open_) in ((a, b, vab, dr in (0, 1)), (b, a, vba, dr in (0, -1))):
            if not open_: continue
            rows.append(u); cols.append(w); tmin.append(ln / 1000 / (v if v > 0 else CAR_FALLBACK_KMH) * 60); lens.append(ln)
    G = coo_matrix((tmin, (rows, cols)), shape=(NC, NC)).tocsr(); GL = coo_matrix((lens, (rows, cols)), shape=(NC, NC)).tocsr()
    _, lab = connected_components(G, directed=True, connection='strong'); giant = np.bincount(lab).argmax(); served = np.array(sorted(v for v in set(rows) | set(cols) if lab[v] == giant))
    return G, GL, served
def path_sums(pred, mats, N):
    valid = pred >= 0; idx_ = np.where(valid)[0]; acc = {}
    for k, M in mats.items():
        e = np.zeros(N); e[idx_] = np.asarray(M[pred[idx_], idx_]).ravel(); acc[k] = e
    jump = np.where(valid, pred, np.arange(N))
    for _ in range(40):
        for k in acc: acc[k] = acc[k] + acc[k][jump]
        nj = jump[jump]
        if np.array_equal(nj, jump): break
        jump = nj
    return acc
def skim_sources(G, mats, src, dst, chunk=40):
    N = G.shape[0]; res = {'gc': np.full((len(src), len(dst)), np.nan)}; res.update({k: np.full_like(res['gc'], np.nan) for k in mats})
    for s0 in range(0, len(src), chunk):
        srcs = src[s0:s0 + chunk]; D, P = dijkstra(G, indices=srcs, directed=True, return_predecessors=True)
        for i in range(len(srcs)):
            res['gc'][s0 + i] = D[i, dst]; sums = path_sums(P[i], mats, N)
            for k in mats: res[k][s0 + i] = sums[k][dst]
    for k in res: res[k][~np.isfinite(res['gc'])] = np.nan
    return res
ac = pd.read_csv('Output/lrt_v2/lrt_area_representative_station.csv', index_col=0).reindex(AREAS)
CAR = {}
for p, cfg in PERIODS.items():
    G, GL, served = car_graph(cfg['car_hour']); kd = cKDTree(CXY[served])
    cd, ci = kd.query(TC); anchor = served[ci]; R = skim_sources(G, {'km': GL}, anchor, anchor)
    car_t = R['gc'] + (cd[:, None] + cd[None, :]) / 1000 / CAR_CONNECTOR_KMH * 60; np.fill_diagonal(car_t, np.nan); CAR[p] = {'ivt': car_t, 'km': R['km'] / 1000, 'gc': car_t + CAR_TERMINAL_MIN}
    cda, cia = kd.query(ac[['cx', 'cy']].values); anc_a = served[cia]; D = dijkstra(G, indices=anc_a, directed=True)[:, anc_a] + (cda[:, None] + cda[None, :]) / 1000 / CAR_CONNECTOR_KMH * 60
    Dd = pd.DataFrame(D, index=AREAS, columns=AREAS)
    for o, d_, lab in [(201, 210, 'Tirat Carmel → Hamifrats'), (210, 217, 'Hamifrats → Nazareth'), (201, 217, 'Tirat Carmel → Nazareth')]:
        for a_, b_, dirn in [(o, d_, 'forward'), (d_, o, 'reverse')]:
            route_rows.append({'mode': f'Car (May 2026 speeds, {cfg["car_hour"]:02d}:00)', 'period': p, 'direction': dirn, 'from': names[a_], 'to': names[b_], 'trips': np.nan, 'stops': np.nan, 'scheduled_min (median)': Dd.loc[a_, b_] + CAR_TERMINAL_MIN, 'observed_min (step 30 ratio)': np.nan})
    print(f'{p}: car TAZ skim done ({np.isfinite(car_t).sum():,} pairs; corridor TAZ median {np.nanmedian(car_t):.1f} min)')
route_times = pd.DataFrame(route_rows); route_times.to_csv(f'{OUT}/time_on_route.csv', index=False, float_format='%.1f'); print(route_times.round(1).to_string(index=False))

AM: car TAZ skim done (30,102 pairs; corridor TAZ median 26.0 min)


PM: car TAZ skim done (30,102 pairs; corridor TAZ median 26.2 min)
                                                                                                      mode period               direction                              from                                to  trips  stops  scheduled_min (median)  observed_min (step 30 ratio)
                                                                                   Metronit line 1 (83001)     AM                       0 ת. מרכזית חוף הכרמל/רציפים עירוני       מרכזית הקריות/הורדה מטרונית   36.0   42.0                    65.0                          57.2
                                                                                   Metronit line 1 (83001)     AM                       1       מרכזית הקריות/איסוף מטרונית ת. מרכזית חוף הכרמל/הורדה מטרונית   36.0   41.0                    65.0                          57.2
                                                              Train (Israel Railways, study-area stations)     

## 3. Transit level of service per period, and the LRT access, egress and line legs

The period's line graph (step 44's construction: stops, line nodes, ride edges at the schedule's
median times with step 30's observed times where they exist — for the PM the AM observed ÷
scheduled ratio per segment is applied to the PM schedule — boarding at 2 × wait + 8, walk
transfers within 300 m, TAZ access to stops within 1 km). Every LRT station of the alternative
is a node reached on foot from stops within 400 m and from TAZ centroids within 1 km (the nearest
station in any case). The **access leg** from each TAZ to each station and the **egress leg** from
each station to each TAZ are shortest paths on this graph with their components. Each end uses
its **cheapest-access station** (step 31's gateway rule — a joint choice of the station pair would
let a long trip ride the Metronit the whole way and the LRT one stop, which is not an LRT trip), and
the LRT cost of a TAZ pair is `access + boarding penalty (8 after a bus, 4 after the Metronit, 0 on
foot) + 2 × 2.5 wait + station access (0.5 per end) + LRT time between the two stations +
egress + alighting penalty`; the pair has no LRT alternative when both ends share the station. The
bus alternative of the pivot is the best no-LRT path (bus or Metronit).

In [5]:
def build_transit(p):
    cfg = PERIODS[p]; st = ST[p].copy(); st['line'] = st['route_code'].astype(int).astype(str) + '_' + st['direction_id'].astype(int).astype(str)
    st = st.sort_values(['trip_id', 'stop_sequence']); g = st.groupby('trip_id'); st['next_stop'] = g['stop_id'].shift(-1); st['next_arr'] = g['arr'].shift(-1); st['dt_min'] = (st['next_arr'] - st['dep']) / 60
    rides = st.dropna(subset=['next_stop']); rides = rides[rides['dt_min'] > 0].copy(); rides['next_stop'] = rides['next_stop'].astype(int)
    rt = rides.groupby(['line', 'stop_id', 'next_stop']).agg(sched_min=('dt_min', 'median'), is_brt=('is_brt', 'max')).reset_index()
    rt['obs_min'] = [seg_obs.get((a, b), np.nan) for a, b in zip(rt['stop_id'], rt['next_stop'])]
    if p == 'AM': rt['ride_min'] = rt['obs_min'].where(rt['obs_min'].notna(), rt['sched_min'])
    else:
        am = ST['AM'].copy(); am['line'] = am['route_code'].astype(int).astype(str) + '_' + am['direction_id'].astype(int).astype(str); am = am.sort_values(['trip_id', 'stop_sequence']); ga = am.groupby('trip_id')
        am['next_stop'] = ga['stop_id'].shift(-1); am['dt'] = (ga['arr'].shift(-1) - am['dep']) / 60; am = am.dropna(subset=['next_stop']); am = am[am['dt'] > 0]
        am_s = am.groupby(['stop_id', 'next_stop'])['dt'].median(); am_s.index = [(int(a), int(b)) for a, b in am_s.index]
        ratio = pd.Series({k: seg_obs[k] / am_s[k] for k in seg_obs.index if k in am_s.index and am_s[k] > 0})
        rt['ratio'] = [ratio.get((a, b), np.nan) for a, b in zip(rt['stop_id'], rt['next_stop'])]; rt['ride_min'] = np.where(rt['ratio'].notna(), rt['sched_min'] * rt['ratio'].clip(0.5, 2.5), rt['sched_min'])
    dep = st.groupby(['line', 'stop_id']).agg(n_peak=('peak', 'sum'), n_win=('trip_id', 'nunique')).reset_index()
    dep['per_hour'] = np.where(dep['n_peak'] > 0, dep['n_peak'], dep['n_win'] / (cfg['hours'][1] - cfg['hours'][0])); dep['wait_min'] = np.minimum(60 / dep['per_hour'] / 2, WAIT_CAP)
    STOPS = sorted(set(rt['stop_id']) | set(rt['next_stop'])); spos = {s: i for i, s in enumerate(STOPS)}; NS = len(STOPS)
    ln_keys = sorted(set(zip(rt['line'], rt['stop_id'])) | set(zip(rt['line'], rt['next_stop']))); lpos = {k: NS + i for i, k in enumerate(ln_keys)}; NL = len(ln_keys)
    Eg = {k: [] for k in ('u', 'v', 'gc', 'ivt', 'walk', 'wait', 'board', 'brt')}
    def add(u, v, gc, ivt=0.0, walk=0.0, wait=0.0, board=0.0, brt=0.0):
        Eg['u'].append(u); Eg['v'].append(v); Eg['gc'].append(gc); Eg['ivt'].append(ivt); Eg['walk'].append(walk); Eg['wait'].append(wait); Eg['board'].append(board); Eg['brt'].append(brt)
    for r in rt.itertuples(): add(lpos[(r.line, r.stop_id)], lpos[(r.line, r.next_stop)], r.ride_min, ivt=r.ride_min, brt=r.ride_min if r.is_brt else 0.0)
    wait_of = dep.set_index(['line', 'stop_id'])['wait_min']
    for (line, s), ln in lpos.items():
        w = wait_of.get((line, s), WAIT_CAP); add(spos[s], ln, W_WAIT * w + TRANSFER_PEN, wait=w, board=1.0); add(ln, spos[s], 0.0)
    sxy = stops_sa.loc[STOPS, ['x', 'y']].values; kd = cKDTree(sxy)
    for i, j in kd.query_pairs(TRANSFER_M):
        wk = np.hypot(*(sxy[i] - sxy[j])) * DETOUR / WALK_SPEED; add(i, j, W_WALK * wk, walk=wk); add(j, i, W_WALK * wk, walk=wk)
    return {'E': Eg, 'add': add, 'NS': NS, 'NL': NL, 'sxy': sxy, 'kd': kd, 'n_rides': len(rt), 'obs_share': rt['obs_min'].notna().mean() if p == 'AM' else rt['ratio'].notna().mean(), 'lines': rt['line'].nunique()}

def finish_graph(T, stations_xy):
    """add TAZ origin / destination nodes and the station nodes, return the csr matrices and the node ranges"""
    NS, NL, add, kd, sxy = T['NS'], T['NL'], T['add'], T['kd'], T['sxy']; O0 = NS + NL; D0 = O0 + NT; K0 = D0 + NT; NK = len(stations_xy); N = K0 + NK
    # TAZ <-> stops
    for ti in range(NT):
        c = TC[ti]; near = kd.query_ball_point(c, ACCESS_M) or [kd.query(c)[1]]
        for i in near:
            wk = np.hypot(*(sxy[i] - c)) * DETOUR / WALK_SPEED; add(O0 + ti, i, W_WALK * wk, walk=wk); add(i, D0 + ti, W_WALK * wk, walk=wk)
    # stations <-> stops, TAZ <-> stations
    for k, (x, y) in enumerate(stations_xy):
        for i in kd.query_ball_point((x, y), STATION_STOP_M):
            wk = np.hypot(sxy[i][0] - x, sxy[i][1] - y) * DETOUR / WALK_SPEED; add(i, K0 + k, W_WALK * wk, walk=wk); add(K0 + k, i, W_WALK * wk, walk=wk)
    kds = cKDTree(stations_xy)
    for ti in range(NT):
        c = TC[ti]; near = kds.query_ball_point(c, ACCESS_M) or [kds.query(c)[1]]
        for k in near:
            wk = np.hypot(stations_xy[k][0] - c[0], stations_xy[k][1] - c[1]) * DETOUR / WALK_SPEED; add(O0 + ti, K0 + k, W_WALK * wk, walk=wk); add(K0 + k, D0 + ti, W_WALK * wk, walk=wk)
    Eg = T['E']; G = csr_matrix((Eg['gc'], (Eg['u'], Eg['v'])), shape=(N, N)); G.data = np.maximum(G.data, 1e-6)
    mats = {k: csr_matrix((Eg[k], (Eg['u'], Eg['v'])), shape=(N, N)) for k in ('ivt', 'walk', 'wait', 'board', 'brt')}
    return G, mats, O0, D0, K0, NK

SK = {}
for p in PERIODS:
    t0 = time.time(); SK[p] = {}
    for alt in ALTS:
        T = build_transit(p); sxy_st = STATIONS[alt][['x', 'y']].values.astype(float)
        G, mats, O0, D0, K0, NK = finish_graph(T, sxy_st); N = G.shape[0]
        orig = np.arange(O0, O0 + NT); dest = np.arange(D0, D0 + NT); stn = np.arange(K0, K0 + NK)
        bus = skim_sources(G, mats, orig, dest)                                                   # no-LRT best path TAZ -> TAZ
        acc = skim_sources(G, mats, orig, stn)                                                    # access legs TAZ -> station
        Gt = G.T.tocsr(); mats_t = {k: M.T.tocsr() for k, M in mats.items()}
        egr = skim_sources(Gt, mats_t, dest, stn)                                                 # egress legs station -> TAZ, as paths on the reversed graph
        for R in (bus, acc, egr): R['gc'] = R['gc'] - TRANSFER_PEN * (R['board'] >= 1)           # the first boarding is not a transfer
        bus_gc = bus['gc'].copy(); np.fill_diagonal(bus_gc, np.nan); brt_based = (bus['brt'] >= 0.5 * np.maximum(bus['ivt'], 1e-9)) & (bus['ivt'] > 0)
        SK[p][alt] = {'bus_gc': bus_gc, 'bus': bus, 'brt_based': brt_based, 'acc': acc, 'egr': egr, 'NK': NK, 'graph': (T['NS'], T['NL'], T['n_rides'], T['obs_share'], T['lines'])}
        print(f"{p} / {ALTS[alt]}: {T['NS']:,} stops, {T['lines']:,} line-directions, {T['n_rides']:,} ride edges ({T['obs_share']:.0%} with step-30 times), {NK} LRT stations; no-LRT paths {np.isfinite(bus_gc).sum():,} of {NT * (NT - 1):,} TAZ pairs, "
              f"Metronit-based {int((brt_based & np.isfinite(bus_gc)).sum()):,}; access legs finite {np.isfinite(acc['gc']).mean():.0%}; {time.time() - t0:.0f} s")

def lrt_skim(p, alt, reg):
    """TAZ-pair LRT generalized cost and components for the alternative and regime, with the boarding / alighting stations"""
    S = SK[p][alt]; acc, egr = S['acc'], S['egr']; st = STATIONS[alt]; T = T_LINE[(alt, reg)].values; NK = S['NK']
    acc_term = np.full(NK, ACC_STATION)
    def pen(R):   # boarding / alighting penalty of a leg: 8 after a bus ride, 4 after the Metronit, 0 on foot
        brt_leg = (R['brt'] >= 0.5 * np.maximum(R['ivt'], 1e-9)) & (R['ivt'] > 0)
        return np.where(R['board'] >= 1, np.where(brt_leg, BRT_LRT_PEN, TRANSFER_PEN), 0.0)
    A_ = acc['gc'] + pen(acc) + (W_WAIT * LRT_WAIT + acc_term)[None, :]            # NT x NK: origin side cost up to boarding
    B_ = egr['gc'] + pen(egr) + acc_term[None, :]                                  # NT x NK: destination side cost from alighting
    # the gateway rule of step 31: each end boards / alights at its cheapest-access station, the LRT runs between them (a joint
    # optimisation over station pairs would let a trip ride the feeder the whole way and the LRT one stop, which is not an LRT trip)
    Ainf = np.where(np.isfinite(A_), A_, np.inf); Binf = np.where(np.isfinite(B_), B_, np.inf)
    a_o = Ainf.argmin(1); b_d = Binf.argmin(1); a_ok = np.isfinite(Ainf.min(1)); b_ok = np.isfinite(Binf.min(1))
    gc = Ainf[np.arange(NT), a_o][:, None] + T[a_o][:, b_d] + Binf[np.arange(NT), b_d][None, :]
    valid = a_ok[:, None] & b_ok[None, :] & (a_o[:, None] != b_d[None, :]) & INTER; np.fill_diagonal(valid, False)
    gc = np.where(valid, gc, np.nan); a_best = np.where(valid, a_o[:, None], -1); b_best = np.where(valid, b_d[None, :], -1)
    ii = np.arange(NT)[:, None].repeat(NT, 1); jj = np.arange(NT)[None, :].repeat(NT, 0); aa = np.where(a_best >= 0, a_best, 0); bb = np.where(b_best >= 0, b_best, 0)
    comp = {'lrt_ivt': T[aa, bb], 'acc_walk': acc['walk'][ii, aa], 'egr_walk': egr['walk'][jj, bb], 'acc_ivt': acc['ivt'][ii, aa], 'egr_ivt': egr['ivt'][jj, bb], 'acc_board': acc['board'][ii, aa], 'egr_board': egr['board'][jj, bb]}
    for k in comp: comp[k] = np.where(np.isfinite(gc), comp[k], np.nan)
    return gc, a_best, b_best, comp

AM / Main route + extension: 10,977 stops, 1,352 line-directions, 44,347 ride edges (97% with step-30 times), 43 LRT stations; no-LRT paths 30,102 of 30,102 TAZ pairs, Metronit-based 10,491; access legs finite 100%; 18 s


AM / Extension only: 10,977 stops, 1,352 line-directions, 44,347 ride edges (97% with step-30 times), 24 LRT stations; no-LRT paths 30,102 of 30,102 TAZ pairs, Metronit-based 10,389; access legs finite 100%; 36 s


PM / Main route + extension: 10,873 stops, 1,211 line-directions, 40,851 ride edges (96% with step-30 times), 43 LRT stations; no-LRT paths 30,102 of 30,102 TAZ pairs, Metronit-based 10,780; access legs finite 100%; 18 s


PM / Extension only: 10,873 stops, 1,211 line-directions, 40,851 ride edges (96% with step-30 times), 24 LRT stations; no-LRT paths 30,102 of 30,102 TAZ pairs, Metronit-based 10,703; access legs finite 100%; 35 s


## 4. Demand: the pivot per TAZ pair, 2022 and the four forecast sets, AM and PM

Base matrices: AM 2022 from step 22 (`final_2022`, car and transit = bus + rail), PM 2022 from
wave 1b (`three_mode_2022_pm`, car; transit = bus + rail); the forecast AM sets from step 23;
the forecast PM sets as the PM base scaled by step 23's AM growth per TAZ pair (origin-row growth
where the AM base cell is empty). The pivot is step 31's, per TAZ pair, with the bus alternative's
cost the best no-LRT path and the LRT premium 5 against a bus-based path, 2.5 against a
Metronit-based one. Taxi is carried unchanged and not reported. The transit of a pair is counted
as **Metronit** where its best no-LRT path rides the Metronit for at least half its in-vehicle
time, otherwise as **bus** (rail is inside that bus figure; it is small in the corridor).

In [6]:
def rdm(path): m_ = pd.read_csv(path, index_col=0); m_.columns = m_.columns.astype(int); m_.index = m_.index.astype(int); return m_.reindex(index=TAZS, columns=TAZS).fillna(0.0).values
BASE = {}
BASE[('AM', '2022')] = {'car': rdm('Output/final_2022/car_2022_taz.csv'), 'transit': rdm('Output/final_2022/transit_2022_taz.csv')}
BASE[('PM', '2022')] = {'car': rdm('Output/ths2017/three_mode_2022_pm/car_2022_taz.csv'), 'transit': rdm('Output/ths2017/three_mode_2022_pm/bus_2022_taz.csv') + rdm('Output/ths2017/three_mode_2022_pm/rail_2022_taz.csv')}
for sc in SCEN_YEARS[1:]:
    BASE[('AM', sc)] = {k: rdm(f'Output/forecast_taz/{sc}/{k}_{sc}_taz.csv') for k in ('car', 'transit')}
    BASE[('PM', sc)] = {}
    for k in ('car', 'transit'):
        b22, f_ = BASE[('AM', '2022')][k], BASE[('AM', sc)][k]
        with np.errstate(divide='ignore', invalid='ignore'):
            g_cell = np.where(b22 > 0, f_ / b22, np.nan); g_row = np.where(b22.sum(1) > 0, f_.sum(1) / b22.sum(1), 1.0)[:, None].repeat(NT, 1)
        g = np.where(np.isfinite(g_cell), g_cell, g_row); BASE[('PM', sc)][k] = BASE[('PM', '2022')][k] * np.clip(g, 0, 5)
for (p, sc), B in BASE.items():
    for k in B: B[k] = B[k] * INTER
print('base trips on the inter-area TAZ pairs (car / transit): ' + '; '.join(f"{p} {sc} {B['car'].sum():,.0f} / {B['transit'].sum():,.0f}" for (p, sc), B in BASE.items()))

def to_area(M): out = np.zeros((A, A)); np.add.at(out, (tai[:, None].repeat(NT, 1), tai[None, :].repeat(NT, 0)), np.nan_to_num(M)); return out
def pivot_run(p, sc, alt, reg):
    B = BASE[(p, sc)]; T_car, T_tr = B['car'], B['transit']; n_pair = T_car + T_tr
    S = SK[p][alt]; bus_gc = S['bus_gc']; brt_based = S['brt_based']
    lrt_gc, a_b, b_b, comp = lrt_skim(p, alt, reg)
    F_car, F_tr = to_area(T_car), to_area(T_tr); S_area = np.where(F_car + F_tr > 0, F_tr / np.where(F_car + F_tr > 0, F_car + F_tr, 1), np.nan); p0 = F_tr.sum() / max((F_car + F_tr).sum(), 1e-9)
    S_prior = np.nan_to_num(S_area[np.ix_(tai, tai)], nan=p0); S_raw = np.where(n_pair > 0, T_tr / np.where(n_pair > 0, n_pair, 1), np.nan)
    S_piv = np.where(n_pair > 0, (n_pair * np.nan_to_num(S_raw) + K_EB * S_prior) / (n_pair + K_EB), np.nan)
    prem = np.where(brt_based, PREM_BRT, PREM_BUS); dL = lrt_gc - prem - bus_gc; avail = np.isfinite(dL) & INTER & (n_pair > 0)
    with np.errstate(over='ignore', invalid='ignore'):
        P_L = np.where(avail, 1 / (1 + np.exp(LAM_T * np.nan_to_num(dL))), 0.0); delta = np.where(avail, -(1 / LAM_T) * np.log1p(np.exp(-LAM_T * np.nan_to_num(dL))), 0.0)
        S_new = np.where(avail, S_piv * np.exp(-LAM * delta) / (S_piv * np.exp(-LAM * delta) + 1 - S_piv), S_piv)
    growth = np.where(avail, np.nan_to_num(S_new) / np.where(np.nan_to_num(S_piv) > 0, S_piv, 1), 1.0)
    T_tr_new = np.minimum(T_tr * growth, n_pair); T_car_new = n_pair - T_tr_new; T_lrt = T_tr_new * P_L; T_other = T_tr_new - T_lrt
    T_brt = np.where(brt_based, T_other, 0.0); T_bus = np.where(brt_based, 0.0, T_other); brt_before = np.where(brt_based, T_tr, 0.0)
    return {'T_car': T_car, 'T_tr': T_tr, 'T_car_new': T_car_new, 'T_tr_new': T_tr_new, 'T_lrt': T_lrt, 'T_bus': T_bus, 'T_brt': T_brt, 'brt_before': brt_before, 'from_car': T_lrt - T_tr * P_L, 'P_L': P_L,
            'lrt_gc': lrt_gc, 'bus_gc': bus_gc, 'a': a_b, 'b': b_b, 'comp': comp, 'avail': avail}

def line_loads(r, alt):
    """LRT trips loaded on the line's station-to-station segments, by direction (1 = towards Nazareth / S24 end, 2 = towards Tirat Carmel)"""
    st = STATIONS[alt]; order = st['order'].values; nseg = len(st) - 1; L1 = np.zeros(nseg); L2 = np.zeros(nseg); board = np.zeros(len(st)); alight = np.zeros(len(st))
    T, a_, b_ = r['T_lrt'], r['a'], r['b']; ok = (T > 0) & (a_ >= 0)
    for i, j in zip(*np.where(ok)):
        t, oa, ob = T[i, j], order[a_[i, j]], order[b_[i, j]]; board[a_[i, j]] += t; alight[b_[i, j]] += t
        if ob > oa: L1[oa:ob] += t
        else: L2[ob:oa] += t
    seg = pd.DataFrame({'from_station': st['station_id'].values[:-1], 'to_station': st['station_id'].values[1:], 'dir1_towards_Nazareth_end': L1, 'dir2_towards_TiratCarmel': L2})
    stn = pd.DataFrame({'station': st['station_id'].values, 'line': st['line'].values, 'TAZ': st['TAZ'].values, 'AggCode': st['AggCode'].values, 'boardings': board, 'alightings': alight})
    return seg, stn

RUNS = {}; summary_rows = []; t0 = time.time()
for p in PERIODS:
    for sc in SCEN_YEARS:
        for alt in ALTS:
            for reg in REGIMES:
                r = pivot_run(p, sc, alt, reg); RUNS[(p, sc, alt, reg)] = r
                d = f'{OUT}/{p}/{sc}/{alt}_{reg}'; os.makedirs(d, exist_ok=True)
                from_other = r['T_lrt'] - r['from_car']; _bb = SK[p][alt]['brt_based']; r['from_brt'] = np.where(_bb, from_other, 0.0); r['from_bus'] = np.where(_bb, 0.0, from_other)   # the shift to the LRT by source: car, Metronit-based or bus-based transit
                for k in ('T_car_new', 'T_tr_new', 'T_lrt', 'T_bus', 'T_brt', 'from_car', 'from_bus', 'from_brt'):
                    pd.DataFrame(r[k], index=TAZS, columns=TAZS).to_csv(f'{d}/{k.lower()}_taz.csv.gz', float_format='%.2f', compression='gzip')
                    pd.DataFrame(to_area(r[k]), index=AREAS, columns=AREAS).to_csv(f'{d}/{k.lower()}_area_v2.csv', float_format='%.1f')
                pd.DataFrame(r['T_car'], index=TAZS, columns=TAZS).to_csv(f'{d}/t_car_base_taz.csv.gz', float_format='%.2f', compression='gzip'); pd.DataFrame(r['T_tr'], index=TAZS, columns=TAZS).to_csv(f'{d}/t_tr_base_taz.csv.gz', float_format='%.2f', compression='gzip')
                seg, stn = line_loads(r, alt); seg.round(1).to_csv(f'{d}/lrt_line_loads.csv', index=False); stn.round(1).to_csv(f'{d}/lrt_stations_boardings.csv', index=False)
                summary_rows.append({'period': p, 'scenario': sc, 'alternative': ALTS[alt], 'regime': REGIMES[reg], 'car': r['T_car_new'].sum(), 'bus': r['T_bus'].sum(), 'BRT (Metronit)': r['T_brt'].sum(), 'LRT': r['T_lrt'].sum(),
                                     'total transit': r['T_tr_new'].sum(), 'LRT from bus/BRT': (r['T_lrt'] - r['from_car']).sum(), 'LRT from car': r['from_car'].sum(), 'car before': r['T_car'].sum(), 'transit before': r['T_tr'].sum(), 'BRT before': r['brt_before'].sum(),
                                     'transit share before': r['T_tr'].sum() / max((r['T_car'] + r['T_tr']).sum(), 1e-9), 'transit share after': r['T_tr_new'].sum() / max((r['T_car'] + r['T_tr']).sum(), 1e-9), 'LRT share of transit': r['T_lrt'].sum() / max(r['T_tr_new'].sum(), 1e-9),
                                     'LRT trips on the extension stations (both ends on S01–S24)': float(r['T_lrt'][(r['a'] >= 0) & (r['a'] < 24) & (r['b'] >= 0) & (r['b'] < 24)].sum()),
                                     'busiest segment load (3 h, one direction)': float(seg[['dir1_towards_Nazareth_end', 'dir2_towards_TiratCarmel']].values.max()), 'TAZ pairs with LRT': int(r['avail'].sum())})
summary = pd.DataFrame(summary_rows); summary.to_csv(f'{OUT}/demand_summary.csv', index=False, float_format='%.1f')
print(f'{len(RUNS)} runs in {time.time() - t0:.0f} s'); print(summary[['period', 'scenario', 'alternative', 'regime', 'car', 'bus', 'BRT (Metronit)', 'LRT', 'total transit', 'LRT from car', 'busiest segment load (3 h, one direction)']].round(0).to_string(index=False))

base trips on the inter-area TAZ pairs (car / transit): AM 2022 56,445 / 14,133; PM 2022 66,517 / 11,607; AM BU_2040 74,670 / 18,321; PM BU_2040 86,451 / 15,610; AM BU_2050 86,417 / 20,652; PM BU_2050 100,598 / 17,818; AM HS_2040 78,027 / 18,787; PM HS_2040 88,649 / 15,737; AM HS_2050 89,560 / 21,232; PM HS_2050 96,724 / 16,329


40 runs in 30 s
period scenario            alternative                                                                       regime      car    bus  BRT (Metronit)    LRT  total transit  LRT from car  busiest segment load (3 h, one direction)
    AM     2022 Main route + extension Prioritized (priority at interchanges; step 25 underground-calibrated times)  55882.0 4096.0          5408.0 5193.0        14696.0         295.0                                     1840.0
    AM     2022 Main route + extension                               Unprioritized (at grade; step 25 ground times)  55999.0 4357.0          6013.0 4209.0        14579.0         210.0                                     1522.0
    AM     2022         Extension only Prioritized (priority at interchanges; step 25 underground-calibrated times)  56023.0 4767.0          5904.0 3884.0        14555.0         187.0                                     1313.0
    AM     2022         Extension only                               Unprior

In [7]:
# skims per period / alternative / regime on the trunk TAZ pairs, for the record, and the no-LRT bus skim
sk_rows = []
on_ext = np.isin(ta, [a for a in AREAS if 201 <= a <= 210]); trunk = on_ext[:, None] & on_ext[None, :] & INTER
for p in PERIODS:
    W = BASE[(p, '2022')]['transit']
    for alt in ALTS:
        S = SK[p][alt]; w = W * trunk; ok = np.isfinite(S['bus_gc']) & (w > 0)
        sk_rows.append({'period': p, 'alternative': ALTS[alt], 'regime': '—', 'mode': 'bus / Metronit best path (no LRT)', 'trunk TAZ pairs, transit-trip-weighted GC': np.average(S['bus_gc'][ok], weights=w[ok]), 'all corridor pairs, transit-trip-weighted GC': np.average(S['bus_gc'][np.isfinite(S['bus_gc']) & (W > 0)], weights=W[np.isfinite(S['bus_gc']) & (W > 0)])})
        for reg in REGIMES:
            r = RUNS[(p, '2022', alt, reg)]; g = r['lrt_gc']; ok = np.isfinite(g) & (w > 0); ok2 = np.isfinite(g) & (W > 0)
            sk_rows.append({'period': p, 'alternative': ALTS[alt], 'regime': REGIMES[reg], 'mode': 'LRT (access + line + egress)', 'trunk TAZ pairs, transit-trip-weighted GC': np.average(g[ok], weights=w[ok]), 'all corridor pairs, transit-trip-weighted GC': np.average(g[ok2], weights=W[ok2])})
    wc = BASE[(p, '2022')]['car'] * INTER; okc = np.isfinite(CAR[p]['gc']) & (wc > 0)
    sk_rows.append({'period': p, 'alternative': '—', 'regime': '—', 'mode': 'car (network + 3 min)', 'trunk TAZ pairs, transit-trip-weighted GC': np.average(CAR[p]['gc'][okc & trunk], weights=wc[okc & trunk]), 'all corridor pairs, transit-trip-weighted GC': np.average(CAR[p]['gc'][okc], weights=wc[okc])})
    pd.DataFrame(CAR[p]['gc'], index=TAZS, columns=TAZS).to_csv(f'{OUT}/{p}/car_gc_taz.csv.gz', float_format='%.2f', compression='gzip')
    for alt in ALTS:
        pd.DataFrame(SK[p][alt]['bus_gc'], index=TAZS, columns=TAZS).to_csv(f'{OUT}/{p}/transit_gc_no_lrt_taz_{alt}.csv.gz', float_format='%.2f', compression='gzip')
        for reg in REGIMES: pd.DataFrame(RUNS[(p, '2022', alt, reg)]['lrt_gc'], index=TAZS, columns=TAZS).to_csv(f'{OUT}/{p}/lrt_gc_taz_{alt}_{reg}.csv.gz', float_format='%.2f', compression='gzip')
skims = pd.DataFrame(sk_rows); skims.to_csv(f'{OUT}/skims_summary.csv', index=False, float_format='%.1f'); print(skims.round(1).to_string(index=False))

period            alternative                                                                       regime                              mode  trunk TAZ pairs, transit-trip-weighted GC  all corridor pairs, transit-trip-weighted GC
    AM Main route + extension                                                                            — bus / Metronit best path (no LRT)                                       44.8                                          54.4
    AM Main route + extension Prioritized (priority at interchanges; step 25 underground-calibrated times)      LRT (access + line + egress)                                       48.7                                          65.1
    AM Main route + extension                               Unprioritized (at grade; step 25 ground times)      LRT (access + line + egress)                                       57.2                                          72.2
    AM         Extension only                                                   

In [8]:
# figures: the two lines with their stations; the demand totals; the line loads of the central combination
fig, ax = plt.subplots(figsize=(11, 7)); tzg.boundary.plot(ax=ax, color='#e1e0d9', lw=.3)
s1 = STATIONS['main_ext']; ax.plot(s1[s1.line == 'extension']['x'], s1[s1.line == 'extension']['y'], color=BLUE, lw=2.5, label='extension (hf_lrt_3), 24 stations')
ax.plot(main_nodes['x'], main_nodes['y'], color=ORANGE, lw=2.5, label=f'main route (Main_Nofit), {len(st_main)} assumed stations'); ax.scatter(s1['x'], s1['y'], s=12, color=INK, zorder=5)
for _, r in s1.iloc[::3].iterrows(): ax.annotate(r['station_id'], (r['x'], r['y']), fontsize=6, color=INK2)
ax.set_xlim(190000, 236000); ax.set_ylim(728000, 752000); ax.set_axis_off(); ax.legend(frameon=False, loc='lower right'); ax.set_title('The two lines: extension (Tirat Carmel – Hamifrats) and main route (Hamifrats – Nazareth)', fontsize=10)
plt.tight_layout(); plt.savefig(f'{FIG}/lrt_alternatives_lines.png', dpi=150); plt.close()
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, p in zip(axes, PERIODS):
    sub = summary[summary['period'] == p]; xs = np.arange(len(SCEN_YEARS)); w_ = .2
    for k, ((alt, al), (reg, rl)) in enumerate([((a, ALTS[a]), (g, REGIMES[g])) for a in ALTS for g in REGIMES]):
        v = [sub[(sub['scenario'] == sc) & (sub['alternative'] == al) & (sub['regime'] == rl)]['LRT'].iloc[0] for sc in SCEN_YEARS]
        ax.bar(xs + (k - 1.5) * w_, v, w_, color=[BLUE, '#86b6ef', ORANGE, '#f2c14e'][k], label=f'{al}, {rl.split(" (")[0]}')
    ax.set_xticks(xs); ax.set_xticklabels(SCEN_YEARS); ax.set_title(f'LRT trips, {PERIODS[p]["label"]}', fontsize=10); ax.grid(axis='y', color=GRID, lw=.6)
    for sp in ('top', 'right'): ax.spines[sp].set_visible(False)
axes[0].set_ylabel('LRT trips in the three-hour period'); axes[0].legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.savefig(f'{FIG}/lrt_alternatives_demand.png', dpi=150); plt.close()
fig, ax = plt.subplots(figsize=(13, 4.5)); seg, _ = line_loads(RUNS[('AM', 'BU_2040', 'main_ext', 'prioritized')], 'main_ext'); xs = np.arange(len(seg))
ax.bar(xs - .2, seg['dir1_towards_Nazareth_end'], .4, color=BLUE, label='towards Nazareth'); ax.bar(xs + .2, seg['dir2_towards_TiratCarmel'], .4, color=ORANGE, label='towards Tirat Carmel')
ax.set_xticks(xs[::2]); ax.set_xticklabels(seg['from_station'].iloc[::2], rotation=90, fontsize=7); ax.set_ylabel('LRT trips on the segment, AM 3 h'); ax.set_title('Line loads, main route + extension, Prioritized, BU 2040, AM', fontsize=10); ax.legend(frameon=False); ax.grid(axis='y', color=GRID, lw=.6)
for sp in ('top', 'right'): ax.spines[sp].set_visible(False)
plt.tight_layout(); plt.savefig(f'{FIG}/lrt_alternatives_line_loads.png', dpi=150); plt.close(); print('figures written')

figures written


## 5. Results

**Times on route.** Extension S01 → S24: 40.6 min Prioritized, 65.8 Unprioritized; main route
S24 → Nazareth at 80 km/h with its 20 stops: 33.7 min (72.5 km/h commercial); through line
Tirat Carmel → Nazareth 74.3 / 99.6 min. Metronit line 1 Hof HaCarmel – Krayot: 65 min scheduled,
57 observed (step 30's ratio 0.88), in both periods; train Hof HaCarmel – Kiryat Motzkin 30–31 min;
car at the 07:00 speeds Tirat Carmel → Hamifrats 32.5 min (20.8 the other way, the Haifa core's
one-way system), Hamifrats → Nazareth 45.8 (41.9), Tirat Carmel → Nazareth 64.6 (58.9), within a
minute of the same at 17:00 (caveat 24).

**Level of service on the trunk TAZ pairs (AM, transit-trip-weighted generalized minutes).** Bus /
Metronit best path 44.8; LRT 48.7 Prioritized, 57.2 Unprioritized (both alternatives; the main
route adds nothing on the trunk pairs); car 13.8. The LRT is dearer than the best bus path on
the trunk because of its walk at 4 km/h to a sparser set of stations and its boarding penalty
after a feeder; it wins on the pairs the Metronit does not serve directly and on the long hauls.

**Demand, 2022 (three-hour periods, corridor TAZ pairs).** AM: main route + extension **5,193**
LRT trips Prioritized (295 from the car) and 4,209 Unprioritized; extension only **3,884** and
2,921. PM: 4,101 / 3,313 / 3,240 / 2,462. The main route adds about 1,300 AM trips to the
extension's — the Kiryat Ata and Nazareth sides board at its first stations and ride into Haifa —
and the through line's busiest segment moves from the extension (S20–S21 towards Tirat Carmel,
1,313) to the main route's entry into Hamifrats (towards Tirat Carmel, 1,840). Of the 2022
corridor transit market (14,133 AM trips) the LRT takes 27–35 %; the bus loses most of it, the
Metronit 400–1,000 trips, and the car 120–300.

**Demand, 2040 / 2050.** AM, main route + extension, Prioritized: **6,309** (BU 2040), **7,433**
(BU 2050), **7,476** (HS 2040), **9,225** (HS 2050); Unprioritized 4,999 / 5,700 / 6,010 / 7,393;
extension only Prioritized 4,879 / 5,939 / 5,424 / 6,653, Unprioritized 3,595 / 4,242 / 4,008 /
4,893. PM, main route + extension, Prioritized: 5,358 / 6,135 / 5,620 / 5,956; extension only
4,270 / 4,960 / 4,608 / 4,918. The busiest segment of the through line reaches 3,058 trips in
the AM three hours (HS 2050, Prioritized) — about 1,800 in the peak hour at step 27's factor.
The PM is 80–85 % of the AM on the extension and lower on the main route, as its base
transit market is (11,607 against 14,133 trips) and its direction reverses.

**Reading the differences.** Prioritized against Unprioritized is worth 20–25 % of the LRT
trips in every scenario — the extension's 25 minutes end to end, nothing else separating the
regimes. The main route against the extension alone is worth 30–35 % in 2022 and 30–40 % in
the forecast years. Against the previous parameter set (4.8 km/h walking and 1.5 min of
station access per Prioritized end) the slower walk costs the LRT and the bus alike, the
equal access term lifts the Prioritized extension by 8 % and lowers the Unprioritized by 2 %,
and the main route's added demand falls by a fifth, its stations being the farthest to walk to.

## 6. Outputs and limits

**Outputs** (`Output/alternatives/`). `stations_{alt}.csv`, `lrt_station_times_{alt}_{regime}.csv`,
`main_route_nodes.csv`; `time_on_route.csv` (LRT, Metronit, train, car); `skims_summary.csv` and,
per period, the TAZ-level car, no-LRT transit and LRT generalized-cost matrices; per
`{period}/{scenario}/{alternative}_{regime}/`: the car, transit, LRT, bus and Metronit trip
matrices at TAZ (`*_taz.csv.gz`) and area level (`*_area_v2.csv`), the line loads by segment and
direction and the boardings by station; `demand_summary.csv` (every run); the workbook and the
Word report are written by the companion script `tools/build_alternatives_report.py`.

**Limits.** The main route's stations are assumed (a); its demand would move with the real
station list and with any feeder restructuring in Nazareth and Kiryat Ata. The forecast PM sets
are the PM base grown at the AM rates. The LRT premium over the Metronit is an estimate from
in-vehicle multipliers, not an estimated parameter. The pivot's induced trips depend on the
aggregation level (caveat 27) and the TAZ-level transit cost is dearer than the area skim's
(caveat 26) — both read on the LRT and bus sides alike here, since both are built on the same
TAZ graph, which is the consistency step 39 lacked. No capacity, no route choice against
parallel bus services (task C7's truncation is not applied), no fare, no park-and-ride.